# Final Project

Author: Evan Whitfield

Course: ST 554

Purpose: Final Project

In [96]:
import pandas as pd
import numpy as np

from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, VectorAssembler, PCA
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col

In [44]:
power_ml_data = pd.read_csv('power_ml_data.csv')

In [45]:
spark_sess = SparkSession.builder.getOrCreate()

spark_df = spark_sess.createDataFrame(power_ml_data)

In [46]:
spark_df.show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
|      5.853|    76.9|     0.081|       

In [47]:
spark_df.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



In [48]:
cast_hour_double = SQLTransformer(statement = 
    """
    SELECT *, CAST(Hour AS DOUBLE) AS Hour_double
    FROM __THIS__
    """
)

In [49]:
binary_day_night = Binarizer(threshold=6.5, inputCol="Hour_double", outputCol="day_night")

In [50]:
one_hot = OneHotEncoder(inputCols=["Month"],outputCols=["Month_ohe"],dropLast=False)

In [66]:
weather_assembled = VectorAssembler(
    inputCols=["Temperature","Humidity","Wind_Speed","General_Diffuse_Flows","Diffuse_Flows"],
    outputCol="weather"
)

In [67]:
pca = PCA(
    k=2,
    inputCol="weather",
    outputCol="pca_features"
)

In [68]:
label_maker = SQLTransformer(statement="""
    SELECT *, Power_Zone_3 AS label
    FROM __THIS__
    """
)

In [76]:
features_assembler = VectorAssembler(
    inputCols=["pca_features","day_night","Power_Zone_1","Power_Zone_2","Month_ohe"],
    outputCol="features"
)

In [77]:
lr = LinearRegression(
    featuresCol="features",
    labelCol="label"
)

In [71]:
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

In [72]:
evaluator = RegressionEvaluator()

In [101]:
pipeline = Pipeline(stages = [
    cast_hour_double, 
    binary_day_night, 
    one_hot, 
    weather_assembled,
    pca,
    label_maker,
    features_assembler,
    lr
])

In [102]:
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=5,
    parallelism = 128
)

In [103]:
model = cv.fit(spark_df)

26/04/28 23:38:49 WARN Instrumentation: [a65c5c22] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 23:38:49 WARN Instrumentation: [c23d6f24] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 23:38:49 WARN Instrumentation: [8baf1f06] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 23:38:49 WARN Instrumentation: [3ffb503f] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 23:38:49 WARN Instrumentation: [53a2eb5a] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 23:38:49 WARN Instrumentation: [ea12cc34] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 23:38:49 WARN Instrumentation: [c6d2ea50] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 23:38:49 WARN Instrumentation: [49049a16] regParam is zero, which might cause numerical instability and overf

In [104]:
print("Best Model Params:", model.bestModel.extractParamMap())

Best Model Params: {}


In [98]:
rmse_values = cv_model.avgMetrics
best_rmse = min(rmse_values)

print("Best CV RMSE:", best_rmse)

Best CV RMSE: 2147.8562108447586


In [90]:
predictions = model.transform(spark_df)

In [94]:
predictions = predictions.withColumn(
    "residual",
    col("label") - col("prediction")
)

In [95]:
result_df = predictions.select(
    "label",
    "prediction",
    "residual"
)

result_df.show()

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386|20869.401445788437| -628.437585788437|
|20131.08434|18652.841100299505| 1478.243239700496|
|19668.43373|18197.627914300767|1470.8058156992338|
|18899.27711| 17583.85396919725| 1315.423140802748|
|18442.40964|16990.847346659015|1451.5622933409868|
|18130.12048|16511.497340059486| 1618.623139940515|
|17945.06024|16087.288361522884| 1857.771878477115|
|17459.27711|15716.945587131437|1742.3315228685624|
|17025.54217|15265.543079153545|1759.9990908464551|
|16794.21687|14933.036128294012| 1861.180741705988|
|16638.07229|14647.193004302075|1990.8792856979253|
|16395.18072|14409.841682954091|1985.3390370459092|
|16117.59036| 14078.05633938908|2039.5340206109195|
| 15822.6506|13620.298686557251|2202.3519134427497|
|15672.28916|13445.834865288052| 2226.454294711948|
|15597.10843| 13297.79451275365|2299.3139172463507|
|15510.36145

Best CV RMSE: 2147.8562108447586
